In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType
from pyspark.sql import functions as F

catalogue_name = 'ecommerce'

In [0]:
# 1. change Quantity colmn to have either 1,2,3.. or one,two,three..
# 2. Remove extra character from unit_price column and keep only amount
# 3. remove % from discount_pct column and show percent in decimal pointer
# 4. Descriptive names for Channel column


In [0]:
brnz_order_items = spark.table(f"{catalogue_name}.bronze.brnz_order_items")
brnz_order_items.show(5)
print(brnz_order_items.printSchema())

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|        11|         10%|         2|    web|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|       1|                GBP|         5|          4%|         1|    web|       NULL|dbf

In [0]:
# 1. change Quantity colmn to have either 1,2,3 type values

# brnz_order_items.select("quantity").distinct().display()
df = brnz_order_items.withColumn("quantity", F.when(F.col("quantity") == 'Two', F.lit(2))\
    .otherwise(F.col("quantity")).cast("int"))

# df.select("quantity").distinct().display()


# 2. Remove extra character from unit_price column and keep only amount

# df.select("unit_price").distinct().display()
df = df.withColumn("unit_price", F.regexp_replace(F.col("unit_price"), "[$]",'').cast("double"))
# df.select("unit_price").distinct().display()

# 3. remove % from discount_pct column and show percent in decimal pointer

# df.select("discount_pct").distinct().display()
df = df.withColumn("discount_pct", F.regexp_replace(F.col("discount_pct"), '%','').cast("double") )
# df.select("discount_pct").distinct().display()

# 4. make coupon_code to lower
df = df.withColumn("coupon_code", F.lower(F.trim(F.col("coupon_code"))))
df.select("coupon_code").distinct().display()

# 5. Descriptive names for Channel column

# df.select("channel").distinct().display()
df = df.withColumn("channel", F.when(F.col("channel") == 'web', 'Website')\
    .when(F.col("channel") == 'app', 'Mobile')\
    .otherwise(F.col('channel')))
# df.select("channel").distinct().display()



coupon_code
null
fest20
save50
prime5
new10


In [0]:
display(df.limit(10))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_source_file,_ingested_at
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 09:44:32,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,3,2000000094205,2,AED,229.0,15.0,20,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,4,2000000293790,2,AED,2222.0,0.0,801,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 21:09:10,CUST000000117094,643614,1,2000000271415,3,INR,252.0,10.0,82,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 13:03:57,CUST000000016826,643615,1,2000000050980,1,GBP,71.0,5.0,9,Website,save50,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z
2025-08-01,2025-08-01 17:54:51,CUST000000252796,643616,1,2000000463254,1,USD,6.0,9.0,1,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z


### Converting the datatype

In [0]:
# 1. converting dt string to date
df = df.withColumn('dt', F.to_date(F.col("dt"),'yyyy-MM-dd'))
df.show(5)

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|      11.0|        10.0|         2|Website|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|       1|                GBP|       5.0|         4.0|         1|Website|       NULL|dbf

In [0]:
# 2. Converting order_ts string to timestamp
df = df.withColumn("order_ts", 
                   F.coalesce(
                       F.to_timestamp(F.col("order_ts"), "yyyy-MM-dd HH:mm:ss"),
                       F.to_timestamp(F.col("order_ts"),"dd-MM-yyyy HH:mm"))
)
df.show(5)

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|      11.0|        10.0|         2|Website|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|       1|                GBP|       5.0|         4.0|         1|Website|       NULL|dbf

In [0]:
# 3. converting item_seq string to int

df =df.withColumn("item_seq", F.col("item_seq").cast("int"))
df.show(5)

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|      11.0|        10.0|         2|Website|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|       1|                GBP|       5.0|         4.0|         1|Website|       NULL|dbf

In [0]:
# 4. Convert tax-amount string to double and strip non-numeric characters

df = df.withColumn("tax_amount", F.regexp_replace(F.col("tax_amount"),r'[^0-9].\-','').cast("double"))
df.show(5)

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|      11.0|        10.0|       2.0|Website|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|       1|                GBP|       5.0|         4.0|       1.0|Website|       NULL|dbf

In [0]:
# Add processed time
df =df.withColumn("processed_time", F.current_timestamp())
df.show(5)

+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+--------------------+
|        dt|           order_ts|     customer_id|order_id|item_seq|   product_id|quantity|unit_price_currency|unit_price|discount_pct|tax_amount|channel|coupon_code|        _source_file|        _ingested_at|      processed_time|
+----------+-------------------+----------------+--------+--------+-------------+--------+-------------------+----------+------------+----------+-------+-----------+--------------------+--------------------+--------------------+
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       1|2000000028279|       1|                GBP|      11.0|        10.0|       2.0|Website|       NULL|dbfs:/Volumes/eco...|2026-08-05 11:07:...|2026-08-05 11:24:...|
|2025-08-01|2025-08-01 22:53:52|CUST000000241190|  643611|       2|2000000377445|   

In [0]:
display(df.limit(10))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_source_file,_ingested_at,processed_time
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,3,2000000094205,2,AED,229.0,15.0,20.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,4,2000000293790,2,AED,2222.0,0.0,801.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T21:09:10.000Z,CUST000000117094,643614,1,2000000271415,3,INR,252.0,10.0,82.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T13:03:57.000Z,CUST000000016826,643615,1,2000000050980,1,GBP,71.0,5.0,9.0,Website,save50,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z
2025-08-01,2025-08-01T17:54:51.000Z,CUST000000252796,643616,1,2000000463254,1,USD,6.0,9.0,1.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-08-05T11:07:30.607Z,2026-08-05T11:24:07.606Z


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.silver.slvr_order_items")